In [9]:
from pathlib import Path

import pandas as pd
import numpy as np

# League standings
The purpose of this notebook is to use our datasets to extract the final season's standings and see if it is consistent with real life results. This helps us check if our data is valid.

In [8]:
raw_data_folder = Path.cwd().parent/"data"/"raw"/"ligue1"

raw_l1_data = {}
for file in raw_data_folder.glob("*.csv"):
    raw_l1_data[file.stem] = pd.read_csv(file)
    print(f"Extracted {file.stem} with shape {raw_l1_data[file.stem].shape}")

Extracted F1_2122 with shape (380, 105)
Extracted F1_2223 with shape (380, 105)
Extracted F1_2324 with shape (306, 105)
Extracted F1_2425 with shape (306, 119)
Extracted F1_2526 with shape (305, 131)


# Number of teams check
## Recovering the Number of Teams from the Number of Matches

For a double round-robin league with \(n\) teams, each team plays every other team twice (home and away).

The total number of matches is therefore:

$$
n(n-1)=k
$$

where \(k\) is the number of matches in the dataset.

Rearranging:

$$
n^2 - n - k = 0
$$

Using the quadratic formula:

$$
n = \frac{1 \pm \sqrt{1+4k}}{2}
$$

Since the number of teams must be a strictly positive integer, we keep only the positive solution:

$$
n = \frac{1 + \sqrt{1+4k}}{2}
$$

For example, if the dataset contains \(380\) matches:

$$
n = \frac{1 + \sqrt{1+4 \times 380}}{2}
= \frac{1 + \sqrt{1521}}{2}
= \frac{1 + 39}{2}
= 20
$$

Therefore, the league contains \(20\) teams. In the test below, we are going to check if the number we get from the formula is the same as the unique values of home and away teams in the data

In [21]:
def get_nb_teams(match_count: int) -> int:
    n = (1+np.sqrt(1+4*match_count))/2
    if not np.isclose(n, round(n)):
        print(
            f"{match_count} matches does not correspond to a complete double round-robin season."
        )

    return int(round(n))

In [22]:
get_nb_teams(raw_l1_data['F1_2122'].shape[0])

20

In [23]:
for name, df in raw_l1_data.items():
    n = get_nb_teams(df.shape[0])
    print(f"{name} has k={df.shape[0]} rows, therefore we get {n} teams.")
    unique_teams = len(set(df['HomeTeam']) | set(df['AwayTeam']))
    print(f"{name} has {unique_teams} unique teams.")
    if unique_teams == n:
        print("Test Worked")
    else:
        print("Problem")

F1_2122 has k=380 rows, therefore we get 20 teams.
F1_2122 has 20 unique teams.
Test Worked
F1_2223 has k=380 rows, therefore we get 20 teams.
F1_2223 has 20 unique teams.
Test Worked
F1_2324 has k=306 rows, therefore we get 18 teams.
F1_2324 has 18 unique teams.
Test Worked
F1_2425 has k=306 rows, therefore we get 18 teams.
F1_2425 has 18 unique teams.
Test Worked
305 matches does not correspond to a complete double round-robin season.
F1_2526 has k=305 rows, therefore we get 18 teams.
F1_2526 has 18 unique teams.
Test Worked


We see that we have one missing match potentially from the last dataset

In [20]:
raw_l1_data['F1_2526']['AwayTeam'].value_counts().add(
    raw_l1_data['F1_2526']['HomeTeam'].value_counts()
).sort_values()

AwayTeam
Nantes        33
Toulouse      33
Brest         34
Angers        34
Lens          34
Lille         34
Lorient       34
Auxerre       34
Lyon          34
Marseille     34
Metz          34
Monaco        34
Nice          34
Paris FC      34
Paris SG      34
Le Havre      34
Rennes        34
Strasbourg    34
Name: count, dtype: int64

### Toulouse VS Nantes 25/26 cancelled
We can see from the data that this match was cancelled since we are missing this match. The link for info https://ligue1.com/fr/articles/l1_article_5119-j34-nantes-toulouse-definitivement-arrete?.

# Points rules
- Win = 3pts
- Draw = 1pt
- Loss = 0pts

Then we have first tie breaker is goal scored - goals conceded

In [51]:
test_df = raw_l1_data['F1_2324']

In [52]:
lyon_df = test_df.loc[
    (test_df['HomeTeam'] == 'Lyon') | (test_df['AwayTeam'] == 'Lyon')
]

In [53]:
def get_pts_lyon(df):
    if df['HomeTeam'] == 'Lyon' and df['FTR'] == 'H':
        return 3
    elif df['AwayTeam'] == 'Lyon' and df['FTR'] == 'A':
        return 3
    elif df['FTR'] == 'D':
        return 1
    else:
        return 0

In [54]:
lyon_df['pts_lyon'] = lyon_df.apply(get_pts_lyon, axis = 1)

In [55]:
lyon_df[['HomeTeam','AwayTeam','FTR','pts_lyon']].sum()

HomeTeam    StrasbourgLyonNiceLyonLyonBrestReimsLyonLyonLy...
AwayTeam    LyonMontpellierLyonParis SGLe HavreLyonLyonLor...
FTR                        HADADHHDADAAHHHAHHAHAHAAAADAHHHAAH
pts_lyon                                                   53
dtype: object

In [67]:
def get_points(
    row : pd.Series,
    team : str
) -> int:
    if row['HomeTeam'] == team and row['FTR'] == 'H':
        return 3
    elif row['AwayTeam'] == team and row['FTR'] == 'A':
        return 3
    elif row['FTR'] == 'D':
        return 1
    else:
        return 0

standings2223 = pd.DataFrame()

teams = list(set(raw_l1_data['F1_2223']['HomeTeam']) | set(raw_l1_data['F1_2223']['AwayTeam']))
for team in teams:
    team_df = raw_l1_data['F1_2223'][[
        'HomeTeam',
        'AwayTeam',
        'FTR',
        'FTHG',
        'FTAG'
    ]].loc[
        (raw_l1_data['F1_2223']['HomeTeam'] == team) |
        (raw_l1_data['F1_2223']['AwayTeam'] == team)
    ].copy()
    
    team_df[f"{team}_pts"] = team_df.apply(lambda row: get_points(row, team), axis = 1)
    
    print(team, team_df[f"{team}_pts"].sum())
    
        

Angers 18
Strasbourg 40
Paris SG 85
Nice 58
Clermont 59
Brest 44
Toulouse 48
Nantes 36
Lens 84
Troyes 24
Lille 67
Lorient 55
Lyon 62
Auxerre 35
Marseille 73
Reims 51
Rennes 68
Ajaccio 26
Montpellier 50
Monaco 65


In [66]:
raw_l1_data['F1_2223'].loc[raw_l1_data['F1_2223']['AwayTeam'] == 'Metz']

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,AvgC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA


In [96]:
def get_season_standings(raw_df : pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()
    home = pd.DataFrame({
        "team": df["HomeTeam"],
        "goals_for": df["FTHG"],
        "goals_against": df["FTAG"],
        "points": np.select(
            [df["FTR"] == "H", df["FTR"] == "D"],
            [3, 1],
            default=0
        )
    })

    away = pd.DataFrame({
        "team": df["AwayTeam"],
        "goals_for": df["FTAG"],
        "goals_against": df["FTHG"],
        "points": np.select(
            [df["FTR"] == "A", df["FTR"] == "D"],
            [3, 1],
            default=0
        )
    })

    standings = (
        pd.concat([home, away])
        .groupby("team", as_index=False)
        .agg(
            points=("points", "sum"),
            goals_for=("goals_for", "sum"),
            goals_against=("goals_against", "sum"),
            nb_matches=("team", "count"))
    )

    standings["goal_diff"] = standings["goals_for"] - standings["goals_against"]

    standings = standings.sort_values(
        by=["points", "goal_diff", "goals_for"],
        ascending=False
    )
    
    return standings

In [97]:
seasons_standings = {}

for name, df in raw_l1_data.items():
    seasons_standings[name] = get_season_standings(df)

In [98]:
for name, standings in seasons_standings.items():
    name = name[3:5]+"/"+name[5:]
    print(f"\nSeason {name}")
    print(standings)


Season 21/22
           team  points  goals_for  goals_against  nb_matches  goal_diff
14     Paris SG      86         90             36          38         54
8     Marseille      71         63             38          38         25
10       Monaco      69         65             40          38         25
13         Nice      67         52             36          38         16
16       Rennes      66         82             40          38         42
18   Strasbourg      63         60             43          38         17
7          Lyon      62         66             51          38         15
4          Lens      62         62             48          38         14
12       Nantes      55         55             48          38          7
5         Lille      55         48             48          38          0
2         Brest      48         49             57          38         -8
15        Reims      46         43             44          38         -1
11  Montpellier      43         49   